<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/06_ordenar_resultados_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ordenar una corrida ResNet50 ya terminada

Ejecutá las dos celdas en Colab (no requiere GPU ni volver a entrenar).
Ingresá la carpeta, el integrante y la corrida originales. Conserva el ZIP y los tres mejores
modelos, verifica sus copias y manda a la papelera los auxiliares de esa misma corrida.
No toca otras corridas ni archivos ajenos. Descarga temporalmente los cuatro archivos para
verificarlos; puede tardar por el tamaño de los modelos. Si falta alguno, no limpia nada.


In [ ]:
"""Guardado por ID de carpeta. No crea carpetas en Drive."""
import hashlib
import json
import re
import zipfile
from pathlib import Path


def id_carpeta(enlace):
    texto = enlace.strip()
    coincidencia = re.search(r'/folders/([A-Za-z0-9_-]+)', texto)
    if coincidencia:
        return coincidencia.group(1)
    if re.fullmatch(r'[A-Za-z0-9_-]{10,}', texto):
        return texto
    raise ValueError('Pegá el enlace de una carpeta de Drive (debe contener /folders/).')


class RespaldoDrive:
    def __init__(self, servicio, folder_id, prefijo, estado, media_factory):
        self.servicio = servicio
        self.folder_id = folder_id
        self.prefijo = prefijo
        self.estado = Path(estado)
        self.media_factory = media_factory
        self.datos = json.loads(self.estado.read_text()) if self.estado.exists() else {}
        identidad = {'folder_id': folder_id, 'prefijo': prefijo}
        if self.datos and self.datos['identidad'] != identidad:
            raise ValueError('Esta corrida ya tiene otro destino. Usá otro nombre de CORRIDA.')
        self.datos.setdefault('identidad', identidad)
        self.datos.setdefault('archivos', {})
        meta = servicio.files().get(fileId=folder_id, supportsAllDrives=True,
            fields='id,name,mimeType,trashed,capabilities(canAddChildren)').execute(num_retries=3)
        if meta.get('trashed') or meta['mimeType'] != 'application/vnd.google-apps.folder':
            raise ValueError('El enlace no corresponde a una carpeta activa.')
        if not meta.get('capabilities', {}).get('canAddChildren', False):
            raise PermissionError('No tenés permiso para guardar archivos en esa carpeta.')
        self.nombre = meta['name']
        self._guardar()

    def _guardar(self):
        self.estado.parent.mkdir(parents=True, exist_ok=True)
        temporal = self.estado.with_suffix('.tmp')
        temporal.write_text(json.dumps(self.datos, indent=2), encoding='utf-8')
        temporal.replace(self.estado)

    def _buscar(self, nombre):
        seguro = nombre.replace('\\', '\\\\').replace("'", "\\'")
        q = f"'{self.folder_id}' in parents and trashed = false and name = '{seguro}'"
        resultado = self.servicio.files().list(q=q, spaces='drive', pageSize=100,
            fields='files(id,name),nextPageToken', supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute(num_retries=3)
        if resultado.get('nextPageToken') or len(resultado.get('files', [])) > 1:
            raise RuntimeError(f'Hay archivos duplicados llamados {nombre}. No se crea otro.')
        return next(iter(resultado.get('files', [])), None)

    def subir(self, ruta, relativo):
        ruta = Path(ruta)
        nombre = self.prefijo + '__' + relativo.replace('/', '__')
        with ruta.open('rb') as archivo:
            digest = hashlib.file_digest(archivo, 'md5').hexdigest()
        registro = self.datos['archivos'].get(nombre)
        if registro and registro.get('md5') == digest:
            return False
        if not registro:
            existente = self._buscar(nombre)
            if existente:
                archivo_id = existente['id']
            else:
                archivo_id = self.servicio.files().generateIds(count=1, space='drive').execute(num_retries=3)['ids'][0]
            registro = {'id': archivo_id}
            self.datos['archivos'][nombre] = registro
            # Persistir el ID ANTES de crear: un reintento usa el mismo ID.
            self._guardar()
        archivo_id = registro['id']
        from googleapiclient.errors import HttpError
        try:
            actual = self.servicio.files().get(fileId=archivo_id, supportsAllDrives=True,
                fields='id,parents,trashed,md5Checksum').execute(num_retries=3)
        except HttpError as exc:
            if exc.resp.status != 404:
                raise
            actual = None
        if actual and (actual.get('trashed') or self.folder_id not in actual.get('parents', [])):
            raise RuntimeError('Un archivo de esta corrida se movió o eliminó. Revisá Drive antes de seguir.')
        if actual and actual.get('md5Checksum') == digest:
            registro['md5'] = digest
            self._guardar()
            return False
        media = self.media_factory(str(ruta), mimetype='application/octet-stream',
            chunksize=8 * 1024 * 1024, resumable=True)
        if actual:
            request = self.servicio.files().update(fileId=archivo_id, media_body=media,
                supportsAllDrives=True, fields='id,md5Checksum')
        else:
            request = self.servicio.files().create(body={'id': archivo_id,
                'name': nombre, 'parents': [self.folder_id]}, media_body=media,
                supportsAllDrives=True, fields='id,md5Checksum')
        respuesta = None
        while respuesta is None:
            _, respuesta = request.next_chunk(num_retries=3)
        if respuesta.get('md5Checksum') != digest:
            raise IOError(f'No se pudo verificar la copia de {nombre}. El original sigue en Colab.')
        registro['md5'] = digest
        self._guardar()
        return True

    VARIANTES = ('imagenet_cabeza', 'imagenet_finetuning', 'aleatorio')

    def sincronizar(self, carpeta):
        """Actualizar solo el ZIP y los mejores modelos, incluso durante el entrenamiento."""
        carpeta = Path(carpeta)
        archivo_zip = carpeta / 'metricas_y_graficos.zip'
        temporal = archivo_zip.with_suffix('.tmp')
        with zipfile.ZipFile(temporal, 'w', compression=zipfile.ZIP_DEFLATED) as z:
            for ruta in sorted(carpeta.rglob('*')):
                if ruta.is_file() and ruta not in (archivo_zip, temporal) and ruta.suffix not in ('.keras', '.tmp', '.zip'):
                    z.write(ruta, ruta.relative_to(carpeta).as_posix())
        temporal.replace(archivo_zip)
        cantidad = self.subir(archivo_zip, archivo_zip.name)
        for variante in self.VARIANTES:
            ruta = carpeta / variante / 'mejor.keras'
            if ruta.is_file():
                cantidad += self.subir(ruta, f'{variante}/mejor.keras')
        print(f'Drive: {cantidad} archivo(s) actualizado(s); máximo cuatro por corrida.')

    def finalizar(self, carpeta, actualizar=True):
        """Verificar los cuatro archivos antes de retirar auxiliares de esta corrida."""
        carpeta = Path(carpeta)
        esperados = ['metricas_y_graficos.zip'] + [f'{v}/mejor.keras' for v in self.VARIANTES]
        for relativo in esperados[1:]:
            if not (carpeta / relativo).is_file():
                raise RuntimeError(f'Falta {relativo}; no se limpia Drive.')
        if actualizar:
            self.sincronizar(carpeta)
        # Comprobar el estado remoto aunque la copia esté en la caché local.
        for relativo in esperados:
            nombre = self.prefijo + '__' + relativo.replace('/', '__')
            registro = self.datos['archivos'][nombre]
            remoto = self.servicio.files().get(fileId=registro['id'], supportsAllDrives=True,
                fields='id,parents,trashed,md5Checksum').execute(num_retries=3)
            if remoto.get('trashed') or self.folder_id not in remoto.get('parents', []) or remoto.get('md5Checksum') != registro['md5']:
                raise IOError('No se verificaron los cuatro archivos; no se limpia Drive.')
        with zipfile.ZipFile(carpeta / 'metricas_y_graficos.zip') as z:
            if z.testzip() is not None:
                raise IOError('ZIP inválido; no se limpia Drive.')
            auxiliares = set(z.namelist())
        auxiliares.update(f'{v}/ultimo.keras' for v in self.VARIANTES)
        # Solo nombres conocidos de esta corrida y archivos ya incluidos en el ZIP.
        for relativo in sorted(auxiliares - set(esperados)):
            nombre = self.prefijo + '__' + relativo.replace('/', '__')
            anterior = self._buscar(nombre)
            if anterior:
                self.servicio.files().update(fileId=anterior['id'], body={'trashed': True},
                    supportsAllDrives=True, fields='id').execute(num_retries=3)
                self.datos['archivos'].pop(nombre, None)
                self._guardar()
        print('Corrida finalizada: un ZIP y tres mejores modelos en Drive.')


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import tempfile

INTEGRANTE = input('Integrante de la corrida existente (ej. pia): ').strip()
CORRIDA = input('Corrida existente (ej. resnet50_01): ').strip()
CARPETA_DRIVE = input('URL de la carpeta que contiene esos resultados: ').strip()
if not re.fullmatch(r'[a-z0-9_-]+', INTEGRANTE) or not re.fullmatch(r'[a-z0-9_-]+', CORRIDA):
    raise ValueError('Usá el integrante y la corrida originales: minúsculas, números, guion o guion bajo.')
auth.authenticate_user()
credenciales, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
servicio = build('drive', 'v3', credentials=credenciales, cache_discovery=False)
folder_id = id_carpeta(CARPETA_DRIVE)
with tempfile.TemporaryDirectory() as temporal:
    local = Path(temporal)
    salida = local / 'resultados'
    salida.mkdir()
    respaldo = RespaldoDrive(servicio, folder_id, f'{INTEGRANTE}__{CORRIDA}',
        local / 'estado.json', MediaFileUpload)
    esperados = ['metricas_y_graficos.zip'] + [f'{v}/mejor.keras' for v in respaldo.VARIANTES]
    # Comprobar los cuatro antes de descargar o retirar cualquier auxiliar.
    archivos = {}
    for relativo in esperados:
        nombre = respaldo.prefijo + '__' + relativo.replace('/', '__')
        archivo = respaldo._buscar(nombre)
        if archivo is None:
            raise RuntimeError(f'Falta {nombre}. No se modificó Drive. Terminá la corrida primero.')
        archivos[relativo] = archivo['id']
    for relativo, archivo_id in archivos.items():
        ruta = salida / relativo
        ruta.parent.mkdir(parents=True, exist_ok=True)
        print('Verificando:', relativo)
        with ruta.open('wb') as destino:
            descarga = MediaIoBaseDownload(destino, servicio.files().get_media(fileId=archivo_id, supportsAllDrives=True))
            terminado = False
            while not terminado:
                _, terminado = descarga.next_chunk(num_retries=3)
        with ruta.open('rb') as archivo:
            digest = hashlib.file_digest(archivo, 'md5').hexdigest()
        nombre = respaldo.prefijo + '__' + relativo.replace('/', '__')
        respaldo.datos['archivos'][nombre] = {'id': archivo_id, 'md5': digest}
    # Conserva el ZIP original: no vuelve a empaquetarlo ni requiere entrenar.
    respaldo.finalizar(salida, actualizar=False)
print('Listo. Revisá tu carpeta:', CARPETA_DRIVE)
